# The Bootstrap Particle Filter

Welcome to the sixth interactive notebook of the `digital-twins` repository. 

We tackle the limitations of the Kalman Filter. The Kalman Filter is mathematically elegant, but it *only* works if your system is Linear and Gaussian. 

What happens if your sensor data is highly non-linear? What if the probability distribution splits into two separate possibilities (a bimodal distribution)? A Gaussian curve (which only has one peak) cannot represent this.

### Enter the Particle Filter (Sequential Monte Carlo)
Instead of using a mathematical formula to represent our belief, we use **thousands of tiny simulations (particles)**. 

The algorithm works in three steps (Algorithm 6.4):
1. **Predict (Sample)**: Every particle runs its own simulation forward in time (adding process noise).
2. **Update (Weight)**: We compare each particle's predicted state against the actual sensor measurement. Particles that match the sensor get a *high weight*; particles that disagree get a *low weight*.
3. **Resample**: We play Darwinian evolution. We "kill" the low-weight particles and "clone" the high-weight particles. 

Over time, the surviving swarm of particles perfectly maps the true probability distribution, no matter how weird or non-linear it is!

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

# Ensure Python can find the 'src' directory in the root of the repository

# Import our Particle Filter engine and Visualization tools
from digital_twins.assimilation.particle import BootstrapParticleFilter
from digital_twins.visualization.uncertainty_plots import plot_1d_particle_histogram

### Interactive Learning: The Bimodal Ambiguity

A mobile agent is moving along a 1D road. It doesn't know its exact position. It only has a temperature sensor.

There are **two fireplaces** on the road (at 80m and 200m). If the agent's sensor reads $30^\circ C$, it could be near Fireplace 1 *or* Fireplace 2. A Kalman Filter would average these and guess the agent is in the middle (at 140m) where it is freezing cold! 

**Instructions:**
1. **Time Step Slider**: Drag this slowly from 1 to 16. 
   - At **Step 4**, watch the particles split into *two distinct peaks* (Bimodal). The filter knows it's near a fire, but doesn't know which one!
   - At **Step 16**, the agent passes the second fire, and the ambiguity is broken. Watch the wrong peak instantly die off, and all particles collapse onto the true position.
2. **Number of Particles ($N$)**: Lower this to `50` and watch the filter struggle to track the agent because there aren't enough "guesses" to cover the search space.

In [ ]:
def interactive_bimodal_particle_filter(time_step=1, n_particles=2000):
    # --- Fixed Seed for Reproducible True Reality ---
    np.random.seed(42)
    
    # --- 1. System Parameters ---
    dt = 1.0
    process_noise_std = np.array([1.5, 0.5]) # Pos and Vel noise
    sensor_std = 2.0
    
    # Transition Model
    def transition_model(x, u=None):
        return np.array([x[0] + x[1] * dt, x[1]])
        
    # Nonlinear Measurement Model (Two Fireplaces)
    def measurement_model(x):
        pos = x[0]
        T1 = 30.0 * np.exp(-((pos - 80.0)**2) / (15.0**2)) + 10.0
        T2 = 30.0 * np.exp(-((pos - 200.0)**2) / (15.0**2)) + 10.0
        return max(T1, T2)
        
    # --- 2. Initialize True State & Particle Filter ---
    true_pos = 20.0
    true_vel = 10.0
    
    # We randomize the starting seed for the particles so they jitter when you change N,
    # but keep the 'true trajectory' consistent.
    rng = np.random.default_rng(seed=123)
    initial_particles = np.column_stack((
        rng.uniform(0, 500, n_particles),  # Initial pos guess anywhere from 0-500m
        rng.uniform(0, 50, n_particles)    # Initial vel guess anywhere from 0-50m/s
    ))
    
    pf = BootstrapParticleFilter(n_particles, initial_particles)
    
    # --- 3. Fast-Forward Simulation to the Requested Time Step ---
    current_temp_reading = 10.0
    
    for k in range(1, time_step + 1):
        # Reality moves
        true_pos = true_pos + true_vel * dt + np.random.normal(0, process_noise_std[0])
        true_vel = true_vel + np.random.normal(0, process_noise_std[1])
        
        # Sensor reads
        current_temp_reading = measurement_model([true_pos, true_vel]) + np.random.normal(0, sensor_std)
        
        # PF Cycle
        pf.predict(transition_model, process_noise_std)
        pf.update(current_temp_reading, measurement_model, sensor_std)
        
    # --- 4. Plotting ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 2]}, sharex=True)
    
    # Plot 1: The Temperature Map
    x_space = np.linspace(0, 300, 1000)
    temps = [measurement_model([x, 0]) for x in x_space]
    ax1.plot(x_space, temps, 'k-', linewidth=2, label='Temperature Profile')
    ax1.axvline(80.0, color='orange', linestyle='--', label='Fireplace 1')
    ax1.axvline(200.0, color='red', linestyle='--', label='Fireplace 2')
    ax1.plot(true_pos, current_temp_reading, 'bX', markersize=10, markeredgewidth=3, label=f'Sensor Reading ({current_temp_reading:.1f}°C)')
    ax1.set_title(f"Physical World at Time Step {time_step}", fontweight='bold')
    ax1.set_ylabel("Temperature (°C)")
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: The Particle Histogram
    positions = pf.particles[:, 0] # Extract just the position variable from particles
    plot_1d_particle_histogram(positions, ax=ax2, bins=80, true_state=true_pos, 
                               title=f"Particle Filter Belief (N={n_particles})")
    ax2.set_xlim(0, 300)
    ax2.set_xlabel("1D Road Position (m)")
    
    plt.tight_layout()
    plt.show()

# Create the interactive widget
interact(interactive_bimodal_particle_filter, 
         time_step=IntSlider(value=1, min=1, max=30, step=1, description='Time Step ($k$):', layout={'width':'500px'}),
         n_particles=IntSlider(value=2000, min=50, max=5000, step=50, description='Particles ($N$):', continuous_update=False, layout={'width':'500px'}));